In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

class CPUSLAM:
    def __init__(self, voxel_size=0.1):
        self.voxel_size = voxel_size
        self.map = {}
        self.pose = np.eye(4)  # Initial pose (identity matrix)
        self.pose_history = []  # To store pose history for trajectory plot

    def detect_and_match_features(self, img1, img2):
        orb = cv2.ORB_create()
        kp1, des1 = orb.detectAndCompute(img1, None)
        kp2, des2 = orb.detectAndCompute(img2, None)

        bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
        matches = bf.match(des1, des2)
        matches = sorted(matches, key=lambda x: x.distance)

        points1 = np.float32([kp1[m.queryIdx].pt for m in matches])
        points2 = np.float32([kp2[m.trainIdx].pt for m in matches])

        return points1, points2

    def estimate_pose(self, points1, points2, camera_matrix):
        E, mask = cv2.findEssentialMat(points1, points2, camera_matrix, method=cv2.RANSAC, prob=0.999, threshold=1.0)
        _, R, t, _ = cv2.recoverPose(E, points1, points2, camera_matrix)
        return R, t

    def triangulate_points(self, proj_matrix1, proj_matrix2, points1, points2):
        points4D = cv2.triangulatePoints(proj_matrix1, proj_matrix2, points1.T, points2.T)
        points4D /= points4D[3]  # Normalize
        return points4D[:3].T

    def add_points_to_voxel_grid(self, points3D):
        for point in points3D:
            voxel = tuple((point / self.voxel_size).astype(int))
            self.map[voxel] = self.map.get(voxel, 0) + 1

    def update_pose(self, R, t):
        new_pose = np.eye(4)
        new_pose[:3, :3] = R
        new_pose[:3, 3] = t.flatten()
        self.pose = self.pose @ new_pose  # Update pose with relative motion
        self.pose_history.append(self.pose[:3, 3])  # Store pose for trajectory plot

def main():
    # Camera intrinsic parameters (KITTI example)
    camera_matrix = np.array([[718.856, 0, 607.1928],
                               [0, 718.856, 185.2157],
                               [0, 0, 1]])

    # Paths to KITTI dataset
    dataset_path = "/home/deepfine_project/KITTI_stereo_2015/data_scene_flow/training"
    left_folder = os.path.join(dataset_path, "image_2")
    right_folder = os.path.join(dataset_path, "image_3")
    
    # Get sorted list of image files
    left_images = sorted(os.listdir(left_folder))
    right_images = sorted(os.listdir(right_folder))
    
    # Initialize SLAM system
    slam = CPUSLAM()

    for i in range(len(left_images) - 1):
        # Load consecutive stereo pairs
        img1_left = cv2.imread(os.path.join(left_folder, left_images[i]), cv2.IMREAD_GRAYSCALE)
        img1_right = cv2.imread(os.path.join(right_folder, right_images[i]), cv2.IMREAD_GRAYSCALE)
        img2_left = cv2.imread(os.path.join(left_folder, left_images[i + 1]), cv2.IMREAD_GRAYSCALE)

        if img1_left is None or img1_right is None or img2_left is None:
            print(f"Error: Unable to load images at index {i}")
            continue

        # Feature matching (stereo images)
        points1, points2 = slam.detect_and_match_features(img1_left, img1_right)

        # Triangulate 3D points for current frame
        proj_matrix1 = np.hstack((np.eye(3), np.zeros((3, 1))))
        proj_matrix2 = np.hstack((np.eye(3), np.array([[0.54], [0], [0]])))  # Baseline = 0.54m (KITTI stereo)
        points3D = slam.triangulate_points(proj_matrix1, proj_matrix2, points1, points2)

        # Add 3D points to the voxel grid map
        slam.add_points_to_voxel_grid(points3D)

        # Feature matching (temporal: current vs next frame)
        points1_temporal, points2_temporal = slam.detect_and_match_features(img1_left, img2_left)

        # Pose estimation (relative motion)
        R, t = slam.estimate_pose(points1_temporal, points2_temporal, camera_matrix)
        slam.update_pose(R, t)

        # Output current pose
        print(f"Frame {i}: Pose\n{slam.pose}")

    # Print voxel map summary
    print(f"Voxel grid contains {len(slam.map)} voxels.")

    # Visualize trajectory (camera path)
    trajectory = np.array(slam.pose_history)
    plt.figure()
    plt.plot(trajectory[:, 0], trajectory[:, 1], label="Camera Trajectory")
    plt.xlabel("X (meters)")
    plt.ylabel("Y (meters)")
    plt.title("Camera Trajectory")
    plt.legend()
    plt.show()

    # Visualize 3D points (voxel grid)
    points3D_all = np.array([point for voxel, count in slam.map.items() for point in [np.array(voxel) * slam.voxel_size]])
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(points3D_all[:, 0], points3D_all[:, 1], points3D_all[:, 2], s=1)
    ax.set_xlabel("X (meters)")
    ax.set_ylabel("Y (meters)")
    ax.set_zlabel("Z (meters)")
    ax.set_title("3D Voxel Map")
    plt.show()

if __name__ == "__main__":
    main()